# AeroFlow Components — Data Profiling

This notebook profiles the raw AeroFlow supply chain datasets to understand their structure, data types, missing values and potential data-quality issues before cleaning.

## 1. Import Libraries

In [ ]:
import pandas as pd

In [7]:
parts = pd.read_csv("../data/raw/parts_master.csv")

In [12]:
parts.head()

,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
0,P00001,Electrical,B,2323.45,27,SUP004,Low,No,735.0
1,P00002,Cabin,C,670.84,31,SUP028,Low,No,NaN
2,P00003,Avionics,C,219.44,49,SUP040,Medium,Yes,NaN
3,P00004,Electrical,A,12488.80,33,SUP034,Low,No,NaN
4,P00005,Cabin,C,1319.96,63,SUP024,Medium,No,NaN


In [9]:
parts.shape

(300, 9)

In [10]:
parts.columns

Index(['part_id', 'part_family', 'criticality_class', 'unit_cost',
       'lead_time_days', 'supplier_id_primary', 'supplier_risk_class',
       'is_repairable', 'shelf_life_days'],
      dtype='str')

In [11]:
parts.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   part_id              300 non-null    str    
 1   part_family          300 non-null    str    
 2   criticality_class    300 non-null    str    
 3   unit_cost            300 non-null    float64
 4   lead_time_days       300 non-null    int64  
 5   supplier_id_primary  300 non-null    str    
 6   supplier_risk_class  300 non-null    str    
 7   is_repairable        300 non-null    str    
 8   shelf_life_days      26 non-null     float64
dtypes: float64(2), int64(1), str(6)
memory usage: 21.2 KB


## 2. Missing Value Assessment

In [13]:
parts.isnull().sum()

part_id                  0
part_family              0
criticality_class        0
unit_cost                0
lead_time_days           0
supplier_id_primary      0
supplier_risk_class      0
is_repairable            0
shelf_life_days        274
dtype: int64

In [14]:
parts.groupby("part_family")["shelf_life_days"].count()

part_family
Avionics        0
Cabin           0
Electrical     15
Engine          0
Fasteners       0
Hydraulics     11
LandingGear     0
Structure       0
Name: shelf_life_days, dtype: int64

In [15]:
parts.groupby("part_family").size()

part_family
Avionics       37
Cabin          33
Electrical     41
Engine         32
Fasteners      61
Hydraulics     33
LandingGear    27
Structure      36
dtype: int64

### Investigating Shelf-Life Values

In [16]:
parts[parts["shelf_life_days"].notnull()]

,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
0,P00001,Electrical,B,2323.45,27,SUP004,Low,No,735.0
19,P00020,Hydraulics,C,295.40,46,SUP013,Medium,Yes,950.0
20,P00021,Hydraulics,C,415.37,49,SUP015,Medium,Yes,709.0
36,P00037,Electrical,A,6621.05,37,SUP014,Medium,No,530.0
42,P00043,Electrical,C,1779.33,21,SUP028,Low,No,500.0
57,P00058,Hydraulics,B,1572.49,41,SUP001,Low,Yes,842.0
69,P00070,Electrical,C,747.93,26,SUP027,Medium,No,887.0
73,P00074,Electrical,A,6860.32,22,SUP012,Low,No,1049.0
77,P00078,Electrical,C,1652.36,18,SUP012,Low,No,795.0
111,P00112,Hydraulics,C,226.73,51,SUP025,Medium,Yes,728.0


### Finding: Shelf-Life Data

`shelf_life_days` contains 274 missing values (91.3% of records). Populated values occur only for Electrical and Hydraulics parts, suggesting shelf life is applicable only to selected components rather than being universally required.

The missing values will be retained during profiling and reviewed during the cleaning stage rather than automatically removed or imputed.

## 3. Duplicate Assessment

In [19]:
parts.duplicated().sum()

np.int64(0)

In [21]:
parts["part_id"].duplicated().sum()

np.int64(0)

### Finding: Duplicate Records

No fully duplicated rows were identified in the parts master data. The `part_id` field was also checked independently and all 300 values were unique, confirming it can be used as the unique identifier for each part.

## 4. Categorical Value Assessment

In [22]:
parts["part_family"].unique()

<StringArray>
[ 'Electrical',       'Cabin',    'Avionics',   'Fasteners',   'Structure',
      'Engine',  'Hydraulics', 'LandingGear']
Length: 8, dtype: str

In [23]:
parts["criticality_class"].unique()

<StringArray>
['B', 'C', 'A']
Length: 3, dtype: str

In [24]:
parts["supplier_risk_class"].unique()

<StringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str

In [25]:
parts["is_repairable"].unique()

<StringArray>
['No', 'Yes']
Length: 2, dtype: str

### Finding: Categorical Values

The categorical fields were reviewed for inconsistent labels and formatting. No issues were identified across `part_family`, `criticality_class`, `supplier_risk_class` or `is_repairable`.

## 5. Numeric Value Assessment

In [26]:
parts.describe()

,unit_cost,lead_time_days,shelf_life_days
count,300.000000,300.000000,26.000000
mean,2109.922333,41.116667,774.076923
std,2572.493113,14.105066,168.920792
min,118.180000,12.000000,437.000000
25%,522.257500,32.000000,650.000000
50%,1155.820000,39.000000,798.500000
75%,2705.557500,49.000000,890.000000
max,18478.000000,100.000000,1063.000000


In [27]:
parts[ parts["lead_time_days"] == parts["lead_time_days"].max()]

,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
293,P00294,Cabin,B,3005.67,100,SUP033,High,No,NaN


In [28]:
parts[ parts["unit_cost"] == parts["unit_cost"].max()]

,part_id,part_family,criticality_class,unit_cost,lead_time_days,supplier_id_primary,supplier_risk_class,is_repairable,shelf_life_days
242,P00243,LandingGear,A,18478.0,20,SUP008,Low,Yes,NaN


## 6. Parts Master Profiling Summary

The `parts_master` dataset contains 300 rows and 9 columns. No duplicate rows were found, and each `part_id` is unique.

The categorical columns were also checked and no inconsistent values were identified.

The main issue found was `shelf_life_days`, which is missing for 274 records (91.3%). The available values only appear for some Electrical and Hydraulics parts, which suggests shelf life may not apply to every part. The missing values were therefore left unchanged for now.

The numeric columns were checked for unusual values. The highest lead time was 100 days and the highest unit cost was 18,478. Both records were reviewed and appeared reasonable, so they were kept in the dataset.

Overall, the dataset is in good condition and only requires limited cleaning.